# Análise exploratória – Segurança pública no RJ (ISP-RJ)
Notebook de exploração e validação do pipeline. Execute a partir da raiz do repositório ou deste diretório.

In [5]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name=='notebooks' else pathlib.Path.cwd()))
from src import etl, analise
import pandas as pd
pd.set_option('display.max_columns', 60)

## 1. ETL
Leitura dos brutos, tratamento de `-`, decimais e tipos, e gravação em `data/processed/`.

In [6]:
anual, mensal = etl.run()
anual.shape, mensal.shape

INFO | Anual: 644 linhas; 54 municípios sem nenhum dado publicado (descartados): Aperibé, Areal, Armação dos Búzios, Arraial do Cabo, Bom Jardim ...
INFO | Mensal: 13892 linhas, 92 municípios, 2014-01-01 a 2026-07-01, nulos em indicadores: 27232
INFO | Arquivos salvos em C:\Users\rafae\OneDrive\Área de Trabalho\seguranca-publica-rj\seguranca-publica-rj\data\processed


((266, 58), (13892, 62))

In [7]:
anual.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   fmun_cod                    266 non-null    int64   
 1   fmun                        266 non-null    str     
 2   ano                         266 non-null    int64   
 3   regiao                      266 non-null    category
 4   hom_doloso                  266 non-null    float64 
 5   lesao_corp_morte            266 non-null    float64 
 6   latrocinio                  266 non-null    float64 
 7   cvli                        266 non-null    float64 
 8   hom_por_interv_policial     266 non-null    float64 
 9   letalidade_violenta         266 non-null    float64 
 10  tentat_hom                  266 non-null    float64 
 11  lesao_corp_dolosa           266 non-null    float64 
 12  estupro                     266 non-null    float64 
 13  hom_culposo                 266

## 2. Qualidade dos dados

In [8]:
bruto = etl.ler_csv_isp(etl.ANUAL_RAW)
ind = etl.indicadores(bruto)
sem = bruto[bruto[ind].isna().all(axis=1)]
print('Municípios sem dados (anual):', sem.fmun.nunique())
print(anual.groupby('regiao', observed=True).fmun.nunique())

Municípios sem dados (anual): 54
regiao
Capital                1
Baixada Fluminense    13
Grande Niterói         3
Interior              21
Name: fmun, dtype: int64


In [9]:
n = mensal.isna().sum(); n[n > 0]

feminicidio                  11868
tentativa_feminicidio        11868
pol_militares_mortos_serv     1748
pol_civis_mortos_serv         1748
dtype: int64

## 3. Estatística descritiva

In [10]:
cols = ['letalidade_violenta','hom_por_interv_policial','roubo_rua','total_roubos','total_furtos']
anual[cols].describe().T.assign(cv=lambda d: d['std']/d['mean']).round(2)

,count,mean,std,min,25%,50%,75%,max,cv
letalidade_violenta,266.0,38.77,21.12,3.92,23.08,37.82,50.30,134.92,0.54
hom_por_interv_policial,266.0,5.27,7.34,0.00,0.62,2.63,7.32,45.82,1.39
roubo_rua,266.0,370.45,360.63,1.01,79.15,245.56,551.23,1608.78,0.97
total_roubos,266.0,651.60,616.93,16.16,169.60,428.54,968.78,2885.31,0.95
total_furtos,266.0,651.39,285.10,171.06,443.93,610.22,795.19,1619.11,0.44


### Outliers (regra IQR) em letalidade violenta

In [11]:
q1, q3 = anual.letalidade_violenta.quantile([.25,.75]); lim = q3 + 1.5*(q3-q1)
anual.loc[anual.letalidade_violenta > lim, ['fmun','ano','letalidade_violenta']].sort_values('letalidade_violenta', ascending=False)

,fmun,ano,letalidade_violenta
170,Queimados,2016,134.92
171,Queimados,2017,112.12
95,Japeri,2018,102.92
93,Japeri,2016,100.44
81,Itaguaí,2018,93.72
4,Angra dos Reis,2018,91.81


## 4. RQ1 – Letalidade violenta e intervenção policial por região

In [12]:
t1 = analise.rq1_tabela(anual); t1

,regiao,ano,letalidade_violenta,hom_por_interv_policial,cvli
0,Capital,2014,24.05,3.83,20.22
1,Capital,2015,24.12,4.74,19.38
2,Capital,2016,29.37,7.12,22.25
3,Capital,2017,32.68,8.08,24.60
4,Capital,2018,29.71,8.34,21.36
5,Capital,2019,28.47,10.81,17.67
6,Capital,2020,21.04,6.15,14.89
7,Baixada Fluminense,2014,57.03,3.71,53.32
8,Baixada Fluminense,2015,48.36,2.98,45.38
9,Baixada Fluminense,2016,61.72,3.94,57.78


In [13]:
analise.rq1_figura(t1).show()

In [14]:
anual.groupby('ano').pct_interv_policial.median().round(1)

ano
2014     2.3
2015     2.8
2016     4.9
2017     7.4
2018    13.7
2019    13.4
2020    17.8
Name: pct_interv_policial, dtype: float64

## 5. RQ2 – Roubos de rua × letalidade violenta

In [15]:
base2 = analise.rq2_base(anual)
analise.rq2_correlacoes(base2)

,grupo,n,pearson_r,p_pearson,spearman_rho,p_spearman
0,Todos,38,0.325,0.0468,0.567,0.0002
1,Baixada Fluminense,13,-0.328,0.2738,-0.220,0.4706
2,Grande Niterói,3,0.336,0.7820,0.500,0.6667
3,Interior,21,0.652,0.0014,0.773,0.0000


In [16]:
analise.rq2_figura(base2).show()

## 6. RQ3 – Efeito da pandemia (série mensal)

In [17]:
analise.rq3_tabela(mensal)

,roubo_rua,total_furtos,letalidade_violenta
mes,,,
1,10.5,-4.6,15.7
2,8.4,7.7,-10.1
3,-33.9,-37.6,-3.0
4,-60.0,-60.3,9.2
5,-65.0,-62.6,1.8
6,-56.3,-58.7,-24.1
7,-43.9,-47.0,-18.3
8,-51.2,-43.0,-5.9
9,-42.1,-38.3,-4.6


In [18]:
analise.rq3_figura(mensal).show()

In [19]:
analise.serie_longa_figura(mensal).show()

## 7. Exportar figuras para o artigo

In [21]:
_ = analise.gerar_tudo()